In [1]:

import torch
import sys, os, pdb
import argparse, logging
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
# print(Path(os.path.realpath(__file__)).parents /'src' / 'voxprofile' / 'src' / 'model' / 'age_sex')
target_path = Path.cwd().parent / 'src' / 'voxprofile' / 'src' / 'model' / 'age_sex'

print(target_path)
from voxprofile.src.model.age_sex.wavlm_demographics import WavLMWrapper
from voxprofile.src.model.age_sex.whisper_demographics import WhisperWrapper
# sys.path.append(os.path.join(str(Path(os.path.realpath(__file__)).parents[1])))
# sys.path.append(os.path.join(str(Path(os.path.realpath(__file__)).parents[1]), 'model', 'age_sex'))

# from wavlm_demographics import WavLMWrapper

# define logging console
import logging
logging.basicConfig(
    format='%(asctime)s %(levelname)-3s ==> %(message)s', 
    level=logging.INFO, 
    datefmt='%Y-%m-%d %H:%M:%S'
)

os.environ["MKL_NUM_THREADS"] = "1" 
os.environ["NUMEXPR_NUM_THREADS"] = "1" 
os.environ["OMP_NUM_THREADS"] = "1" 


/home/tiche/Github/Tools4Speech/src/voxprofile/src/model/age_sex


/home/tiche/.conda/envs/vdt/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [2]:
import io
import pandas as pd
from typing import Dict

raw_data = """speaker	start_sec	end_sec	seg_filename	duration_sec	transcription	type
P1	3.02	39.6	../outputs/dyad/P1/P1_seg_0_3.02_39.60_pad0p20.wav	36.58	Jeg kan faktisk godt lide at lave mad...	turn
P1	47.76	48.23	../outputs/dyad/P1/P1_seg_1_47.76_48.23_pad0p20.wav	0.47	Ja.	backchannel
P1	56.22	56.65	../outputs/dyad/P1/P1_seg_2_56.22_56.65_pad0p20.wav	0.43	Ja.	backchannel
P2	0.78	2.6	../outputs/dyad/P2/P2_seg_0_0.78_2.60_pad0p20.wav	1.82	Kan du lide at lave mad?	turn
P2	41.01	59.98	../outputs/dyad/P2/P2_seg_1_41.01_59.98_pad0p20.wav	18.97	Det genkender jeg faktisk vældig godt...	turn"""

df = pd.read_csv(io.StringIO(raw_data), sep="\t")
df["duration_sec"] = df["duration_sec"].round(2)

data = df.to_dict(orient="records") 

segments_by_speaker: Dict[str, pd.DataFrame] = {
    str(speaker): group_df.reset_index(drop=True) 
    for speaker, group_df in df.groupby("speaker")
}
segments_by_speaker

{'P1':   speaker  start_sec  end_sec  \
 0      P1       3.02    39.60   
 1      P1      47.76    48.23   
 2      P1      56.22    56.65   
 
                                         seg_filename  duration_sec  \
 0  ../outputs/dyad/P1/P1_seg_0_3.02_39.60_pad0p20...         36.58   
 1  ../outputs/dyad/P1/P1_seg_1_47.76_48.23_pad0p2...          0.47   
 2  ../outputs/dyad/P1/P1_seg_2_56.22_56.65_pad0p2...          0.43   
 
                               transcription         type  
 0  Jeg kan faktisk godt lide at lave mad...         turn  
 1                                       Ja.  backchannel  
 2                                       Ja.  backchannel  ,
 'P2':   speaker  start_sec  end_sec  \
 0      P2       0.78     2.60   
 1      P2      41.01    59.98   
 
                                         seg_filename  duration_sec  \
 0  ../outputs/dyad/P2/P2_seg_0_0.78_2.60_pad0p20.wav          1.82   
 1  ../outputs/dyad/P2/P2_seg_1_41.01_59.98_pad0p2...         18.97   
 
    

In [2]:
import os
import gc
import soundfile as sf
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from typing import List, Dict, Any, Optional
import pandas as pd

import gc
import numpy as np
import soundfile as sf
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from tqdm.auto import tqdm

import torch
import sys, os, pdb
import argparse, logging
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
# print(Path(os.path.realpath(__file__)).parents /'src' / 'voxprofile' / 'src' / 'model' / 'age_sex')
target_path = Path.cwd().parent / 'src' / 'voxprofile' / 'src' / 'model' / 'age_sex'

print(target_path)
from voxprofile.src.model.age_sex.wavlm_demographics import WavLMWrapper
from voxprofile.src.model.age_sex.whisper_demographics import WhisperWrapper
@dataclass
class TransformersWavLMModel:
    backend: str
    model: Any
    WavLM_model_name: str
    device: str
    cache_dir: Optional[str]
    model_batch_size: int
    compute_type: str

SEX_UNIQUE_LABELS = ["Female", "Male"]

def load_age_sex_model(
    agesex_model_name: str = "tiantiaf/wavlm-large-age-sex",
    device: str = "auto",
    cache_dir: Optional[str] = None,
    model_batch_size: int = 16,
    backend: str = "auto",
    compute_type: Optional[str] = None,
) -> TransformersWavLMModel:
    """Initialise and return a demographic prediction model via Voxprofile."""
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    elif device == "cuda" and not torch.cuda.is_available():
        device = "cpu"

    if compute_type is None:
        compute_type = "float16" if device == "cuda" else "float32"

    if "wavlm" in agesex_model_name:
        backend = "wavlm-large"
        model = WavLMWrapper.from_pretrained(agesex_model_name).to(device)
        model.eval() 
        
        if compute_type == "float16" and device == "cuda":
            model = model.half()

        return TransformersWavLMModel(
            backend=backend,
            model=model,
            WavLM_model_name=agesex_model_name,
            device=device,
            cache_dir=cache_dir,
            model_batch_size=model_batch_size,
            compute_type=compute_type,
        )
    
    raise ValueError(f"Unsupported model or backend: {agesex_model_name}")
def _wavlm_predict_batch_inference(
    batch_segments: List[Dict[str, Any]], 
    model_wrapper: Any
) -> List[Dict[str, Any]]:
    """Loads audio, tracks actual sample lengths, pads them dynamically, and runs batch inference."""
    device = model_wrapper.device
    model = model_wrapper.model
    
    audio_tensors = []
    lengths = []
    max_len = 0
    
    # 1. Load raw audio and track original lengths (From your working snippet)
    for seg in batch_segments:
        audio, sr = sf.read(seg["seg_filename"])
        if audio.ndim > 1:
            audio = audio[:, 0]
            
        tensor = torch.tensor(audio, dtype=torch.float32)
        audio_tensors.append(tensor)
        lengths.append(len(tensor))
        if len(tensor) > max_len:
            max_len = len(tensor)
            
    # 2. Dynamic zero-padding to make shapes uniform for stacking
    padded_tensors = []
    for tensor in audio_tensors:
        pad_size = max_len - len(tensor)
        if pad_size > 0:
            tensor = F.pad(tensor, (0, pad_size), "constant", 0.0)
        padded_tensors.append(tensor)
        
    # 3. Stack arrays and lengths into shapes expected by the model
    input_batch = torch.stack(padded_tensors).to(device)
    input_lengths = torch.tensor(lengths, dtype=torch.long).to(device)
    
    if getattr(model_wrapper, "compute_type", None) == "float16" and device == "cuda":
        input_batch = input_batch.half()
        
    # 4. Model Inference Execution with lengths
    with torch.no_grad():
        # Passing length=input_lengths directly to fix the masking IndexError!
        outputs = model(input_batch, length=input_lengths)
    return outputs


def _batch_files(
    segments: pd.DataFrame,
    output_dir: str,
    batch_size: Optional[float] = 30.0,
) ->List[List[Dict[str, Any]]]:
    os.makedirs(output_dir, exist_ok=True)
    
    segment_info = []
    for idx, row in segments.iterrows():
        seg_filename = row.get("seg_filename", f"segment_{idx}.wav")
        
        segment_info.append({
            "idx": idx,
            "speaker": row.get("speaker", "unknown"),
            "start_sec": float(row["start_sec"]),
            "end_sec": float(row["end_sec"]),
            "seg_filename": seg_filename,
        })

    max_batch_duration = (
        float(batch_size) if batch_size and batch_size > 0 else float("inf")
    )

    batches: List[List[Dict[str, Any]]] = []
    current_batch: List[Dict[str, Any]] = []
    current_duration = 0.0

    for seg in segment_info:
        seg_duration = float(seg["end_sec"] - seg["start_sec"])

        if seg_duration > max_batch_duration:
            if current_batch:
                batches.append(current_batch)
                current_batch = []
                current_duration = 0.0
            batches.append([seg])
            continue

        if current_batch and current_duration + seg_duration > max_batch_duration:
            batches.append(current_batch)
            current_batch = [seg]
            current_duration = seg_duration
        else:
            current_batch.append(seg)
            current_duration += seg_duration

    if current_batch:
        batches.append(current_batch)

    return batches

/home/tiche/Github/Tools4Speech/src/voxprofile/src/model/age_sex


/home/tiche/.conda/envs/vdt/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [5]:
import os
import gc
import torch
import pandas as pd
import torch.nn.functional as F
from typing import Any, Dict, List, Optional
import os
import torch
import pandas as pd
import soundfile as sf
import torch.nn.functional as F
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

@dataclass
class TransformersCharModel:
    backend: str
    model: Any
    Char_model_name: str
    device: str
    cache_dir: Optional[str]
    model_batch_size: int
    compute_type: str

def _batch_files(
    segments: pd.DataFrame,
    output_dir: str,
    batch_size: Optional[float] = 30.0,
) ->List[List[Dict[str, Any]]]:
    os.makedirs(output_dir, exist_ok=True)
    
    segment_info = []
    for idx, row in segments.iterrows():
        seg_filename = row.get("seg_filename", f"segment_{idx}.wav")
        
        segment_info.append({
            "idx": idx,
            "speaker": row.get("speaker", "unknown"),
            "start_sec": float(row["start_sec"]),
            "end_sec": float(row["end_sec"]),
            "seg_filename": seg_filename,
        })

    max_batch_duration = (
        float(batch_size) if batch_size and batch_size > 0 else float("inf")
    )

    batches: List[List[Dict[str, Any]]] = []
    current_batch: List[Dict[str, Any]] = []
    current_duration = 0.0

    for seg in segment_info:
        seg_duration = float(seg["end_sec"] - seg["start_sec"])

        if seg_duration > max_batch_duration:
            if current_batch:
                batches.append(current_batch)
                current_batch = []
                current_duration = 0.0
            batches.append([seg])
            continue

        if current_batch and current_duration + seg_duration > max_batch_duration:
            batches.append(current_batch)
            current_batch = [seg]
            current_duration = seg_duration
        else:
            current_batch.append(seg)
            current_duration += seg_duration

    if current_batch:
        batches.append(current_batch)

    return batches

def _char_predict_batch_inference(
    batch_segments: List[Dict[str, Any]], 
    model_wrapper: Any
) -> List[Dict[str, Any]]:
    """Loads audio, tracks actual sample lengths, pads them dynamically, and runs batch inference."""
    device = model_wrapper.device
    model = model_wrapper.model
    
    audio_tensors = []
    lengths = []
    max_len = 0
    
    # 1. Load raw audio and track original lengths (From your working snippet)
    for seg in batch_segments:
        audio, sr = sf.read(seg["seg_filename"])
        if audio.ndim > 1:
            audio = audio[:, 0]
            
        tensor = torch.tensor(audio, dtype=torch.float32)
        audio_tensors.append(tensor)
        lengths.append(len(tensor))
        if len(tensor) > max_len:
            max_len = len(tensor)
            
    # 2. Dynamic zero-padding to make shapes uniform for stacking
    padded_tensors = []
    for tensor in audio_tensors:
        pad_size = max_len - len(tensor)
        if pad_size > 0:
            tensor = F.pad(tensor, (0, pad_size), "constant", 0.0)
        padded_tensors.append(tensor)
        
    # 3. Stack arrays and lengths into shapes expected by the model
    input_batch = torch.stack(padded_tensors).to(device)
    input_lengths = torch.tensor(lengths, dtype=torch.long).to(device)
    
    if getattr(model_wrapper, "compute_type", None) == "float16" and device == "cuda":
        input_batch = input_batch.half()
        
    # 4. Model Inference Execution with lengths
    with torch.no_grad():
        # Passing length=input_lengths directly to fix the masking IndexError!
        outputs = model(input_batch, length=input_lengths)
    return outputs
        
from voxprofile.src.model.emotion.wavlm_emotion import WavLMWrapper
from voxprofile.src.model.emotion.whisper_emotion import WhisperWrapper
EMOTION_LABELS = [
    'Anger', 'Contempt', 'Disgust', 'Fear', 'Happiness', 
    'Neutral', 'Sadness', 'Surprise', 'Other'
]

def load_SER_model(
    SER_model_name: str = "tiantiaf/wavlm-large-categorical-emotion",
    device: str = "auto",
    cache_dir: Optional[str] = None,
    model_batch_size: int = 16,
    backend: str = "auto",
    compute_type: Optional[str] = None,
) -> TransformersCharModel:
    """Initialise and return a emotion prediction model via Voxprofile."""
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    elif device == "cuda" and not torch.cuda.is_available():
        device = "cpu"

    if compute_type is None:
        compute_type = "float16" if device == "cuda" else "float32"

    if "wavlm" in SER_model_name:
        backend = "wavlm-large"
        model = WavLMWrapper.from_pretrained(SER_model_name).to(device)
        model.eval() 
    elif "whisper" in SER_model_name:
        backend = 'whisper'
        model = WhisperWrapper.from_pretrained(SER_model_name).to(device)
        model.eval() 
    else:
        raise ValueError(f"Unsupported model or backend: {SER_model_name}")
    
    if compute_type == "float16" and device == "cuda":
        model = model.half()
    return TransformersCharModel(
        backend=backend,
        model=model,
        Char_model_name=SER_model_name,
        device=device,
        cache_dir=cache_dir,
        model_batch_size=model_batch_size,
        compute_type=compute_type,
    )
    
    

def predict_emotion_segments(
    model: Any,
    segments: pd.DataFrame,
    output_dir: str,
    cache: bool = True,
    batch_size: Optional[float] = 30.0,
    min_duration_samples: int = 1600,
) -> Dict[str, Any]:
    """
    Slices segments into dynamic batches, verifies disk-cached files, 
    and passes uncached elements to WavLM batch inference before returning results.
    """
    batches = _batch_files(segments, output_dir, batch_size)
    predictions_map = {}

    for batch in batches:
        print(batch)
        files_to_predict = []
        file_indices = []
        batch_results = [None] * len(batch)

        for i, seg in enumerate(batch):
            demo_cache = seg['seg_filename'].replace(".wav", "_emotions.txt")

            if cache and os.path.exists(demo_cache):
                try:
                    with open(demo_cache, "r", encoding="utf-8") as cache_file:
                        cached_text = cache_file.read().strip()
                    if not cached_text.startswith("[EMOTION_PREDICTION_FAILED:"):
                        parts = cached_text.split(" | ")
                        Catemo = float(parts[0].split(": ")[1])
                        batch_results[i] = {"EmoCat": Catemo}
                        continue
                except Exception:
                    pass  

            files_to_predict.append(seg)
            file_indices.append(i)

        # Model Inference execution block 
        if files_to_predict:
            outputs = _char_predict_batch_inference(files_to_predict, model)
            print(outputs)
            emotion_prob = F.softmax(outputs[0], dim=1)
            emo_indices = torch.argmax(emotion_prob, dim=1).cpu().tolist()
            # Map generated data targets back into batch metrics
            for batch_idx, pred_idx in zip(file_indices, emo_indices):
                current_emo = EMOTION_LABELS[pred_idx]

                batch_results[batch_idx] = {
                    "EmoCat": current_emo
                }

                if cache:
                    # Construct cache file path smoothly
                    cache_path = batch[batch_idx].get("emo_cache", batch[batch_idx]['seg_filename'].replace(".wav", "_emotion.txt"))
                    with open(cache_path, "w", encoding="utf-8") as cache_file:
                        cache_file.write(f"EmoCat: {current_emo}")

        # Save results back using their true global identifiers
        for seg, res in zip(batch, batch_results):
            predictions_map[seg["idx"]] = res

        # Clean up system memory after every batch iteration
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return predictions_map

In [6]:
SER_model_batch_size = 10
SER_model_name = "tiantiaf/whisper-large-v3-msp-podcast-emotion"
device = "auto"
model = load_SER_model(
    SER_model_name=SER_model_name,
    device=device,
    model_batch_size=SER_model_batch_size,
)
final_results = []
for speaker, segments in  segments_by_speaker.items():
    print(f"Processing emotions for speaker: {speaker} with {len(segments)} segments")
    predictions_map = predict_emotion_segments(model, segments, output_dir="outputs/emo", cache=False)
    
    for idx, row in segments.iterrows():
        # Look up the prediction using the true pandas DataFrame row index (idx)
        pred = predictions_map.get(idx)
        
        final_results.append({
            "speaker": row["speaker"],
            "start_sec": row["start_sec"],
            "end_sec": row["end_sec"],
            "estimated_EmoCat": pred["EmoCat"],
            "file_source": row.get("seg_filename", f"segment_{idx}.wav")
        })

Some weights of WhisperModel were not initialized from the model checkpoint at openai/whisper-large-v3 and are newly initialized because the shapes did not match:
- encoder.embed_positions.weight: found shape torch.Size([1500, 1280]) in the checkpoint and torch.Size([750, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Processing emotions for speaker: P1 with 3 segments
[{'idx': 0, 'speaker': 'P1', 'start_sec': 3.02, 'end_sec': 39.6, 'seg_filename': '../outputs/dyad/P1/P1_seg_0_3.02_39.60_pad0p20.wav'}]


AttributeError: 'NoneType' object has no attribute '_attn_implementation'

In [1]:
import torch
import logging
import torchaudio
import sys, os, pdb
import torch.nn.functional as F

from pathlib import Path

from voxprofile.src.model.emotion.whisper_emotion import WhisperWrapper


# define logging console
import logging
logging.basicConfig(
    format='%(asctime)s %(levelname)-3s ==> %(message)s', 
    level=logging.INFO, 
    datefmt='%Y-%m-%d %H:%M:%S'
)

os.environ["MKL_NUM_THREADS"] = "1" 
os.environ["NUMEXPR_NUM_THREADS"] = "1" 
os.environ["OMP_NUM_THREADS"] = "1" 

if __name__ == '__main__':

    label_list = [
        'Anger', 
        'Contempt', 
        'Disgust', 
        'Fear', 
        'Happiness', 
        'Neutral', 
        'Sadness', 
        'Surprise', 
        'Other'
    ]

    # Find device
    device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
    if torch.cuda.is_available(): print('GPU available, use GPU')

    # Define the model
    # Note that ensemble yields the better performance than the single model
    model = WhisperWrapper.from_pretrained("tiantiaf/whisper-large-v3-msp-podcast-emotion").to(device)
    model.eval()
    
    # Our training data filters output audio shorter than 3 seconds (unreliable predictions) and longer than 15 seconds (computation limitation)
    # So you need to prepare your audio to a maximum of 15 seconds, 16kHz and mono channel
    data = torch.zeros([1, 16000]).float().to(device)
    logits, embedding, _, _, _, _ = model(
        data, return_feature=True
    )
    
    emotion_prob   = F.softmax(logits, dim=1)

    print(emotion_prob.shape)
    print(embedding.shape)
    print(label_list[torch.argmax(emotion_prob).detach().cpu().item()])
    

Some weights of WhisperModel were not initialized from the model checkpoint at openai/whisper-large-v3 and are newly initialized because the shapes did not match:
- encoder.embed_positions.weight: found shape torch.Size([1500, 1280]) in the checkpoint and torch.Size([750, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AttributeError: 'NoneType' object has no attribute '_attn_implementation'

In [ ]:
import os
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm
import torch.nn.functional as F
from typing import Any, Dict, List, Optional
from voxprofile.src.model.emotion.wavlm_emotion_dim import WavLMWrapper

def load_emo_dim_model(
    emo_dim_model_name: str = "tiantiaf/wavlm-large-msp-podcast-emotion-dim",
    device: str = "auto",
    cache_dir: Optional[str] = None,
    model_batch_size: int = 16,
    backend: str = "auto",
    compute_type: Optional[str] = None,
) -> TransformersWavLMModel:
    """Initialise and return a demographic prediction model via Voxprofile."""
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    elif device == "cuda" and not torch.cuda.is_available():
        device = "cpu"

    if compute_type is None:
        compute_type = "float16" if device == "cuda" else "float32"

    if "wavlm" in emo_dim_model_name:
        backend = "wavlm-large"
        model = WavLMWrapper.from_pretrained(emo_dim_model_name).to(device)
        model.eval() 
        
        if compute_type == "float16" and device == "cuda":
            model = model.half()

        return TransformersWavLMModel(
            backend=backend,
            model=model,
            WavLM_model_name=emo_dim_model_name,
            device=device,
            cache_dir=cache_dir,
            model_batch_size=model_batch_size,
            compute_type=compute_type,
        )
    
    raise ValueError(f"Unsupported model or backend: {emo_dim_model_name}")
def predict_emotion_dim_segments(
    model: Any,
    segments: pd.DataFrame,
    output_dir: str,
    cache: bool = True,
    batch_size: Optional[float] = 30.0,
    min_duration_samples: int = 1600,
) -> Dict[str, Any]:
    """
    Slices segments into dynamic batches, verifies disk-cached files, 
    and passes uncached elements to WavLM batch inference before returning results.
    """
    batches = _batch_files(segments, output_dir, batch_size)
    predictions_map = {}

    for batch in tqdm(batches, desc=f"Processing {len(batches)} demographic batches"):
        files_to_predict = []
        file_indices = []
        batch_results = [None] * len(batch)

        for i, seg in enumerate(batch):
            demo_cache = seg['seg_filename'].replace(".wav", "_demographics.txt")

            if cache and os.path.exists(demo_cache):
                try:
                    with open(demo_cache, "r", encoding="utf-8") as cache_file:
                        cached_text = cache_file.read().strip()
                    if not cached_text.startswith("[EMO_DIM_PREDICTION_FAILED:"):
                        parts = cached_text.split(" | ")
                        arousal = float(parts[0].split(": ")[1])
                        valence = float(parts[1].split(": ")[1])
                        dominance = float(parts[2].split(": ")[1])
                        batch_results[i] = {"arousal": arousal, "valence": valence, "dominance": dominance}
                        continue
                except Exception:
                    pass  

            files_to_predict.append(seg)
            file_indices.append(i)

        # Model Inference execution block 
        if files_to_predict:
            a_head, v_head, d_head = _wavlm_predict_batch_inference(files_to_predict, model)
            v_preds = v_head.detach().cpu().flatten().tolist()
            a_preds = a_head.detach().cpu().flatten().tolist()
            d_preds = d_head.detach().cpu().flatten().tolist()
            
            # Map generated data targets back into batch metrics
            for batch_idx  in file_indices:
                # print(batch_idx, output)
                current_v = float(v_preds[batch_idx])
                current_a = float(a_preds[batch_idx])
                current_d = float(d_preds[batch_idx])

                batch_results[batch_idx] = {
                    "arousal": current_a,
                    "valence": current_v, 
                    "dominance": current_d
                }

                if cache:
                    # Construct cache file path smoothly
                    cache_path = batch[batch_idx].get("demo_cache", batch[batch_idx]['seg_filename'].replace(".wav", "_demographics.txt"))
                    with open(cache_path, "w", encoding="utf-8") as cache_file:
                        cache_file.write(f"arousal: {current_a}, valence: {current_v}, dominance: {current_d}")

        # Save results back using their true global identifiers
        for seg, res in zip(batch, batch_results):
            predictions_map[seg["idx"]] = res

        # Clean up system memory after every batch iteration
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return predictions_map

In [42]:
SER_model_batch_size = 10
SER_model_name = "tiantiaf/wavlm-large-msp-podcast-emotion-dim"
device = "auto"
model = load_emo_dim_model(
    emo_dim_model_name=SER_model_name,
    device=device,
    model_batch_size=SER_model_batch_size,
)
final_results = []
for speaker, segments in  segments_by_speaker.items():
    print(f"Processing emotions for speaker: {speaker} with {len(segments)} segments")
    predictions_map = predict_emotion_dim_segments(model, segments, output_dir="outputs/emo", cache=False)
    
    for idx, row in segments.iterrows():
        # Look up the prediction using the true pandas DataFrame row index (idx)
        pred = predictions_map.get(idx)
        final_results.append({
            "speaker": row["speaker"],
            "start_sec": row["start_sec"],
            "end_sec": row["end_sec"],
            "arousal": pred["arousal"],
            "valence": pred["valence"],
            "dominance": pred["dominance"],
            "file_source": row.get("seg_filename", f"segment_{idx}.wav")
        })

Processing emotions for speaker: P1 with 3 segments


Processing 2 demographic batches:   0%|          | 0/2 [00:00<?, ?it/s]

/home/tiche/.conda/envs/vdt/lib/python3.10/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


0.4166269600391388 0.11204070597887039 0.20839187502861023
0.24335543811321259 0.07316981256008148 0.1456465870141983
0.3224000632762909 0.12978138029575348 0.21156226098537445
0.11204070597887039
0.07316981256008148
0.12978138029575348
Processing emotions for speaker: P2 with 2 segments


Processing 1 demographic batches:   0%|          | 0/1 [00:00<?, ?it/s]

0.32846251130104065 0.06648103892803192 0.17146846652030945
0.3742016851902008 0.08821236342191696 0.19098244607448578
0.06648103892803192
0.08821236342191696
